<a href="https://colab.research.google.com/github/joseph-loeffler/eeg-cs590/blob/main/Deep_Learning_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS590-04 (Deep Learning) - Final Project - EEG
## Setup

In [1]:
!pip install mne pyedflib
!pip install -q mne torch torchvision torchaudio

In [2]:
import os
import numpy as np
import mne
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings(
    "ignore",
    message="Limited .* annotation.* outside the data range"
)
mne.set_log_level('WARNING')

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensures reproducibility (slightly slower)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## Data Processing

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/CS 590-04 (Deep Learning)/Final Project/data"
# Copy data to Colab disk (5-10 mins)
!cp -r "$BASE" /content/

Mounted at /content/drive


In [4]:
DATA_DIR = "/content/data"

RUNS = ["R03", "R04", "R07", "R08", "R11", "R12"]

SFREQ = 160
TMIN = 0.0
TMAX = 2.0   # 2-second windows

BATCH_SIZE = 64

In [5]:
def load_subject(subject_id):
    subject = f"S{subject_id:03d}"
    raws = []

    for run in RUNS:
        path = os.path.join(DATA_DIR, subject, f"{subject}{run}.edf")
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)

        raw.filter(1., 40., verbose=False)
        raws.append(raw)

    raw = mne.concatenate_raws(raws)

    events, event_id = mne.events_from_annotations(raw)

    event_dict = {'T1': event_id['T1'], 'T2': event_id['T2']}

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_dict,
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]

    # Convert labels
    y = (y == event_id['T2']).astype(int)

    target_len = int(SFREQ * (TMAX - TMIN))  # 320

    # --- FORCE uniform length ---
    X_fixed = []

    for x in X:
        if x.shape[1] >= target_len:
            x = x[:, :target_len]   # crop
        else:
            pad = target_len - x.shape[1]
            x = np.pad(x, ((0, 0), (0, pad)))  # pad

        X_fixed.append(x)

    X = np.stack(X_fixed)

    return X, y

In [6]:
def load_subjects(subjects):
    X_all, y_all = [], []

    for s in subjects:
        X, y = load_subject(s)
        X_all.append(X)
        y_all.append(y)

    return np.concatenate(X_all), np.concatenate(y_all)

In [7]:
subjects = list(range(1, 110))

rng = np.random.default_rng(seed=42)
rng.shuffle(subjects)

train_subjects = subjects[:90]
test_subjects = subjects[90:]

print("Train:", train_subjects[:10], "...")
print("Test:", test_subjects)

train_subjects = subjects[:80]
val_subjects   = subjects[80:90]
test_subjects  = subjects[90:]

Train: [34, 19, 69, 29, 107, 1, 25, 79, 101, 89] ...
Test: [99, 12, 94, 42, 23, 20, 36, 65, 48, 13, 54, 15, 14, 67, 37, 98, 66, 78, 9]


In [8]:
def preprocess(X):
    # normalize per sample
    X = (X - X.mean(axis=(1,2), keepdims=True)) / (X.std(axis=(1,2), keepdims=True) + 1e-6)

    # reshape for CNN
    X = X[:, np.newaxis, :, :]

    return X

In [9]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Models

In [10]:
class EEGNet(nn.Module):
    def __init__(self, n_channels=64, n_samples=320, n_classes=2):
        super().__init__()

        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(8, 16, (n_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.25)
        )

        self.separable = nn.Sequential(
            nn.Conv2d(16, 16, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(0.25)
        )

        self.classifier = nn.Linear(16 * (n_samples // 32), n_classes)

    def forward(self, x):
        x = self.firstconv(x)
        x = self.depthwise(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [11]:
class EEG_CNN_3Layer(nn.Module):
    def __init__(self, n_channels=64, n_samples=320, n_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            # Conv Layer 1
            nn.Conv2d(1, 16, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),

            # Conv Layer 2
            nn.Conv2d(16, 32, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),

            # Conv Layer 3
            nn.Conv2d(32, 64, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((1, 2))
        )

        # Compute flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.features(dummy)
            self.flatten_dim = out.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

In [12]:
def train_model(model, train_loader, val_loader, epochs=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # evaluation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds = model(X).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += len(y)

        val_acc = correct / total
        print(f"Epoch {epoch+1} | Loss: {total_loss:.2f} | Val Acc: {val_acc:.3f}")

## Training + Validation Accuracy

In [13]:
X_train, y_train = load_subjects(train_subjects)
X_val, y_val     = load_subjects(val_subjects)
X_test, y_test   = load_subjects(test_subjects)

X_train = preprocess(X_train)
X_val   = preprocess(X_val)
X_test  = preprocess(X_test)

train_ds = EEGDataset(X_train, y_train)
val_ds   = EEGDataset(X_val, y_val)
test_ds  = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# model = EEGNet()
model = EEG_CNN_3Layer()

train_model(model, train_loader, val_loader, epochs=10)

Epoch 1 | Loss: 209.86 | Val Acc: 0.613
Epoch 2 | Loss: 70.89 | Val Acc: 0.780
Epoch 3 | Loss: 64.73 | Val Acc: 0.754
Epoch 4 | Loss: 61.82 | Val Acc: 0.791
Epoch 5 | Loss: 59.58 | Val Acc: 0.814
Epoch 6 | Loss: 59.58 | Val Acc: 0.827
Epoch 7 | Loss: 55.58 | Val Acc: 0.829
Epoch 8 | Loss: 54.45 | Val Acc: 0.830
Epoch 9 | Loss: 52.21 | Val Acc: 0.819
Epoch 10 | Loss: 51.31 | Val Acc: 0.840


## Final Evaluation

In [ ]:
def evaluate(model, loader):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval()

    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)

    return correct / total

In [15]:
# test_acc = evaluate(model, test_loader)
# print("Final Test Accuracy:", test_acc)